# Max capacities workflow

This notebook is mainly an **analysis-first interface** for max-capacity scenario CSVs already available under `data/`.

Use it to:
- load precomputed max-capacity scenarios
- compare them with quick tables, deltas, and heatmaps
- inspect a primary scenario in more detail
- optionally regenerate one scenario locally when you really need to

## Recommended usage

- **Preferred local use:** load and compare prerun CSVs copied back from ARC.
- **Preferred heavy-build path:** use the ARC land-processing scripts for the expensive preprocessing and then sync the finished CSVs into `data/`.
- **Optional local rebuild:** set `RUN_HEAVY_BUILD = True` in the config cell below. This can take a long time and requires the full local geospatial stack.

## Canonical scenario names

The current ARC workflow produces these named outputs when available locally:
- `baseline_slope15`
- `baseline_allslopes`
- `paper_2pct_slope15`
- `high_50pct_slope15`

The legacy convenience alias `data/max_capacities.csv` may also be present as `legacy_alias` in the scenario table below.

In [ ]:
from __future__ import annotations

from pathlib import Path
import importlib
import sys

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for path in [start] + list(start.parents):
        if (path / "model").is_dir():
            return path
    raise RuntimeError("Could not locate repo root containing 'model/'")


NOTEBOOK_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PROJECT_ROOT = find_repo_root(NOTEBOOK_DIR)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from model import land_processing as lp

lp = importlib.reload(lp)

pd.options.display.float_format = "{:,.2f}".format


def scenario_label_from_path(path: Path) -> str:
    if path.name == "max_capacities.csv":
        return "legacy_alias"
    return path.stem.replace("max_capacities_", "", 1)


def discover_scenario_csvs(data_dir: Path) -> dict[str, Path]:
    scenario_files: dict[str, Path] = {}
    for path in sorted(data_dir.glob("max_capacities*.csv")):
        scenario_files[scenario_label_from_path(path)] = path
    return scenario_files


def preferred_scenario_order(labels: list[str]) -> list[str]:
    preferred = [
        "baseline_slope15",
        "baseline_allslopes",
        "paper_2pct_slope15",
        "high_50pct_slope15",
        "legacy_alias",
    ]
    ordered = [label for label in preferred if label in labels]
    ordered.extend(sorted(label for label in labels if label not in ordered))
    return ordered


def summarise_scenario(label: str, path: Path, df: pd.DataFrame) -> dict[str, object]:
    positive_mask = df["max_capacity_mw"] > 0
    onshore = df["onshore_land_pct"] if "onshore_land_pct" in df.columns else pd.Series(np.nan, index=df.index)
    return {
        "scenario": label,
        "csv": str(path.relative_to(PROJECT_ROOT)),
        "rows": int(len(df)),
        "positive_cells": int(positive_mask.sum()),
        "offshore_only_cells": int(((onshore <= 0) & positive_mask).sum()),
        "total_capacity_gw": float(df["max_capacity_mw"].sum() / 1_000.0),
        "solar_capacity_gw": float(df["max_power_solar_mw"].sum() / 1_000.0),
        "wind_capacity_gw": float(df["max_power_wind_mw"].sum() / 1_000.0),
        "median_cell_capacity_mw": float(df["max_capacity_mw"].median()),
        "max_cell_capacity_gw": float(df["max_capacity_mw"].max() / 1_000.0),
    }


def _cell_id(lat: float, lon: float, decimals: int = 2) -> str:
    return f"{lat:.{decimals}f}_{lon:.{decimals}f}"


def _build_cell_geojson(
    df: pd.DataFrame,
    lat_col: str = "latitude",
    lon_col: str = "longitude",
    id_col: str = "cell_id",
    cell_size_deg: float = 1.0,
) -> dict[str, object]:
    half = cell_size_deg / 2.0
    features = []
    for lat, lon, cell_id in zip(df[lat_col], df[lon_col], df[id_col]):
        lat0 = float(lat)
        lon0 = float(lon)
        polygon = [
            [lon0 - half, lat0 - half],
            [lon0 - half, lat0 + half],
            [lon0 + half, lat0 + half],
            [lon0 + half, lat0 - half],
            [lon0 - half, lat0 - half],
        ]
        features.append(
            {
                "type": "Feature",
                "id": cell_id,
                "properties": {"latitude": lat0, "longitude": lon0},
                "geometry": {"type": "Polygon", "coordinates": [polygon]},
            }
        )
    return {"type": "FeatureCollection", "features": features}


def build_heatmap(
    df: pd.DataFrame,
    value_col: str,
    title: str,
    *,
    color_scale: str = "Viridis",
    color_midpoint: float | None = None,
    cell_size_deg: float = 1.0,
    hover_cols: list[str] | None = None,
):
    plot_df = df.loc[df[value_col].notna()].copy()
    plot_df["cell_id"] = [_cell_id(lat, lon) for lat, lon in zip(plot_df["latitude"], plot_df["longitude"])]
    cell_geojson = _build_cell_geojson(plot_df, cell_size_deg=cell_size_deg)

    hover_cols = hover_cols or []
    hover_data = {column: True for column in hover_cols if column in plot_df.columns}
    hover_data[value_col] = True

    fig = px.choropleth_map(
        plot_df,
        geojson=cell_geojson,
        locations="cell_id",
        featureidkey="id",
        color=value_col,
        map_style="carto-positron",
        color_continuous_scale=color_scale,
        color_continuous_midpoint=color_midpoint,
        opacity=0.85,
        labels={value_col: value_col},
        hover_data=hover_data,
        center=dict(lat=0, lon=0),
        zoom=0.5,
        title=title,
    )
    fig.update_traces(marker_line_width=0.2, marker_line_color="rgba(255, 255, 255, 0.3)")
    fig.update_layout(margin=dict(l=0, r=0, t=40, b=0))
    return fig

In [ ]:
DATA_DIR = PROJECT_ROOT / "data"
LAND_COVER_FILE = DATA_DIR / "MCD12C1.A2022001.061.2023244164746.hdf"
BATHYMETRY_FILE = DATA_DIR / "model_bathymetry.nc"
CELL_SIZE_DEG = 1.0

AVAILABLE_SCENARIO_FILES = discover_scenario_csvs(DATA_DIR)
AVAILABLE_SCENARIOS = preferred_scenario_order(list(AVAILABLE_SCENARIO_FILES))

if not AVAILABLE_SCENARIOS:
    raise FileNotFoundError(
        "No max-capacity CSVs found under data/. Sync precomputed scenarios from ARC or set RUN_HEAVY_BUILD = True below."
    )

PRIMARY_SCENARIO = "baseline_slope15" if "baseline_slope15" in AVAILABLE_SCENARIO_FILES else AVAILABLE_SCENARIOS[0]
SELECTED_SCENARIOS = [
    label
    for label in ["baseline_slope15", "baseline_allslopes", "paper_2pct_slope15", "high_50pct_slope15"]
    if label in AVAILABLE_SCENARIO_FILES
] or AVAILABLE_SCENARIOS[: min(4, len(AVAILABLE_SCENARIOS))]

if PRIMARY_SCENARIO not in SELECTED_SCENARIOS:
    SELECTED_SCENARIOS = [PRIMARY_SCENARIO] + SELECTED_SCENARIOS

COMPARISON_TARGET = next(
    (
        label
        for label in ["baseline_allslopes", "paper_2pct_slope15", "high_50pct_slope15"]
        if label in SELECTED_SCENARIOS and label != PRIMARY_SCENARIO
    ),
    PRIMARY_SCENARIO,
)

MAP_METRIC = "max_capacity_mw"
DIFFERENCE_METRIC = "max_capacity_mw"

# Optional local rebuild path. This is useful for small tests, but the full
# preprocessing is heavy and is usually better submitted through the ARC scripts.
RUN_HEAVY_BUILD = False
HEAVY_BUILD_LABEL = "baseline_slope15"
HEAVY_BUILD_OUTPUT_CSV = DATA_DIR / f"max_capacities_{HEAVY_BUILD_LABEL}.csv"

HEAVY_BUILD_CONFIG = lp.LandAvailabilityConfig(
    land_cover_path=LAND_COVER_FILE,
    bathymetry_path=BATHYMETRY_FILE,
    output_csv=HEAVY_BUILD_OUTPUT_CSV,
)

print("Available local max-capacity scenarios:")
for label in AVAILABLE_SCENARIOS:
    print(f"  - {label:20s} -> {AVAILABLE_SCENARIO_FILES[label].relative_to(PROJECT_ROOT)}")

print()
print("Notebook defaults:")
print(f"  PRIMARY_SCENARIO   = {PRIMARY_SCENARIO}")
print(f"  SELECTED_SCENARIOS = {SELECTED_SCENARIOS}")
print(f"  COMPARISON_TARGET  = {COMPARISON_TARGET}")
print(f"  RUN_HEAVY_BUILD    = {RUN_HEAVY_BUILD}")
print("  Preferred workflow: analyse existing CSVs locally, rebuild heavy scenarios on ARC when needed.")

In [ ]:
selected_scenario_paths: dict[str, Path] = {}

if RUN_HEAVY_BUILD:
    print("RUN_HEAVY_BUILD=True. Regenerating one scenario locally from raw geospatial inputs.")
    print("For the canonical full workflow and scenario matrix runs, prefer the ARC submit/resubmit scripts.")
    built_label = scenario_label_from_path(HEAVY_BUILD_OUTPUT_CSV)
    max_caps_df = lp.write_land_availability_table(HEAVY_BUILD_CONFIG)
    loaded_scenarios = {built_label: max_caps_df}
    selected_scenario_paths[built_label] = HEAVY_BUILD_OUTPUT_CSV
    PRIMARY_SCENARIO = built_label
    SELECTED_SCENARIOS = [built_label]
    COMPARISON_TARGET = built_label
else:
    missing = [label for label in SELECTED_SCENARIOS if label not in AVAILABLE_SCENARIO_FILES]
    if missing:
        raise FileNotFoundError(
            f"Missing local scenario CSVs for {missing}. Sync them from ARC or change SELECTED_SCENARIOS."
        )
    loaded_scenarios = {label: pd.read_csv(AVAILABLE_SCENARIO_FILES[label]) for label in SELECTED_SCENARIOS}
    selected_scenario_paths = {label: AVAILABLE_SCENARIO_FILES[label] for label in SELECTED_SCENARIOS}
    max_caps_df = loaded_scenarios[PRIMARY_SCENARIO]
    print("Analysis mode: loaded precomputed scenario CSVs from data/.")
    print("This is the recommended local notebook mode.")

print()
print(f"Primary scenario in downstream cells: {PRIMARY_SCENARIO}")
print(f"Primary CSV: {selected_scenario_paths[PRIMARY_SCENARIO].relative_to(PROJECT_ROOT)}")
print(f"Rows loaded: {len(max_caps_df):,}")

In [ ]:
scenario_overview = pd.DataFrame(
    summarise_scenario(label, selected_scenario_paths[label], loaded_scenarios[label])
    for label in SELECTED_SCENARIOS
).sort_values("scenario").reset_index(drop=True)

print("Loaded scenario overview:")
scenario_overview

In [ ]:
summary_cols = [
    "max_capacity_mw",
    "max_power_solar_mw",
    "max_power_wind_mw",
    "solar_availability",
    "wind_availability",
    "onshore_land_pct",
    "offshore_sea_pct",
]
summary_cols = [column for column in summary_cols if column in max_caps_df.columns]

primary_summary = max_caps_df[summary_cols].describe(percentiles=[0.5, 0.9, 0.99]).T
print(f"Primary scenario summary: {PRIMARY_SCENARIO}")
display(primary_summary)

top_cols = [
    "latitude",
    "longitude",
    "max_capacity_mw",
    "max_power_solar_mw",
    "max_power_wind_mw",
    "onshore_land_pct",
    "elevation_m",
]
top_cols = [column for column in top_cols if column in max_caps_df.columns]

top_sites = max_caps_df.nlargest(10, "max_capacity_mw")[top_cols]
print()
print(f"Top 10 cells by max capacity in {PRIMARY_SCENARIO}:")
display(top_sites)

comparison_delta_df = None
comparison_aggregate = None
cell_delta_summary = None

if COMPARISON_TARGET != PRIMARY_SCENARIO and COMPARISON_TARGET in loaded_scenarios:
    reference_df = loaded_scenarios[PRIMARY_SCENARIO]
    target_df = loaded_scenarios[COMPARISON_TARGET]

    comparison_delta_df = reference_df[
        ["latitude", "longitude", DIFFERENCE_METRIC, "onshore_land_pct"]
    ].merge(
        target_df[["latitude", "longitude", DIFFERENCE_METRIC, "onshore_land_pct"]],
        on=["latitude", "longitude"],
        how="inner",
        suffixes=("_reference", "_target"),
    )
    comparison_delta_df["metric_delta"] = (
        comparison_delta_df[f"{DIFFERENCE_METRIC}_target"]
        - comparison_delta_df[f"{DIFFERENCE_METRIC}_reference"]
    )

    comparison_aggregate = pd.DataFrame(
        [
            {
                "metric": "Total capacity (GW)",
                PRIMARY_SCENARIO: reference_df["max_capacity_mw"].sum() / 1_000.0,
                COMPARISON_TARGET: target_df["max_capacity_mw"].sum() / 1_000.0,
                "delta": (target_df["max_capacity_mw"].sum() - reference_df["max_capacity_mw"].sum()) / 1_000.0,
            },
            {
                "metric": "Solar capacity (GW)",
                PRIMARY_SCENARIO: reference_df["max_power_solar_mw"].sum() / 1_000.0,
                COMPARISON_TARGET: target_df["max_power_solar_mw"].sum() / 1_000.0,
                "delta": (target_df["max_power_solar_mw"].sum() - reference_df["max_power_solar_mw"].sum()) / 1_000.0,
            },
            {
                "metric": "Wind capacity (GW)",
                PRIMARY_SCENARIO: reference_df["max_power_wind_mw"].sum() / 1_000.0,
                COMPARISON_TARGET: target_df["max_power_wind_mw"].sum() / 1_000.0,
                "delta": (target_df["max_power_wind_mw"].sum() - reference_df["max_power_wind_mw"].sum()) / 1_000.0,
            },
        ]
    )
    cell_delta_summary = pd.DataFrame(
        [
            {
                "cells_higher": int((comparison_delta_df["metric_delta"] > 1e-9).sum()),
                "cells_lower": int((comparison_delta_df["metric_delta"] < -1e-9).sum()),
                "cells_unchanged": int((comparison_delta_df["metric_delta"].abs() <= 1e-9).sum()),
                "largest_positive_delta_mw": float(comparison_delta_df["metric_delta"].max()),
                "largest_negative_delta_mw": float(comparison_delta_df["metric_delta"].min()),
            }
        ]
    )

    print()
    print(f"{COMPARISON_TARGET} minus {PRIMARY_SCENARIO}:")
    display(comparison_aggregate)
    display(cell_delta_summary)
else:
    print()
    print("Only one scenario is loaded, so the delta summary and comparison heatmap are skipped.")

## Optional background: MODIS land-cover suitability factors

You can skip this section if you only want to inspect precomputed scenario CSVs.

These factors explain how the heavy max-capacity build translates MODIS land-cover fractions into wind and solar siting availability. They matter when you regenerate a scenario locally or on ARC, or when you need to interpret why a region's capacity looks high or low.

| Class | Description | Wind | Solar |
|-------|-------------|------|-------|
| 0  | Water bodies              | 1.0  | 0.0  |
| 1–5 | Forests (all types)      | 0.0  | 0.0  |
| 6  | Closed shrublands          | 0.5  | 0.5  |
| 7  | Open shrublands            | 0.5  | 0.5  |
| 8  | Woody savannas             | 0.15 | 0.1  |
| 9  | Savannas                   | 0.2  | 0.2  |
| 10 | Grasslands                 | 0.2  | 0.2  |
| 11 | Permanent wetlands         | 0.0  | 0.0  |
| 12 | Croplands                  | 0.0  | 0.05 |
| 13 | Urban and built-up         | 0.0  | 0.0  |
| 14 | Cropland/veg mosaics       | 0.0  | 0.1  |
| 15 | Snow and ice               | 0.0  | 0.0  |
| 16 | Barren/sparse              | 1.0  | 1.0  |

In [ ]:
# Reference table of the MODIS class dictionaries used in model.land_processing
igbp_names = {
    0: "Water",
    1: "Evergreen needleleaf",
    2: "Evergreen broadleaf",
    3: "Deciduous needleleaf",
    4: "Deciduous broadleaf",
    5: "Mixed forest",
    6: "Closed shrublands",
    7: "Open shrublands",
    8: "Woody savannas",
    9: "Savannas",
    10: "Grasslands",
    11: "Wetlands",
    12: "Croplands",
    13: "Urban",
    14: "Cropland/veg mosaic",
    15: "Snow/ice",
    16: "Barren",
}

factors_df = pd.DataFrame(
    [
        {
            "class": key,
            "name": igbp_names.get(key, f"Class {key}"),
            "wind": lp.MODIS_CLASS_WIND_AVAILABILITY.get(key, 0.0),
            "solar": lp.MODIS_CLASS_SOLAR_AVAILABILITY.get(key, 0.0),
        }
        for key in range(17)
    ]
)
factors_df

In [ ]:
fig_avail = px.scatter(
    max_caps_df,
    x="solar_availability",
    y="wind_availability",
    color="onshore_land_pct",
    color_continuous_scale="Viridis",
    labels={
        "solar_availability": "Solar availability (fraction)",
        "wind_availability": "Wind availability (fraction)",
        "onshore_land_pct": "Onshore %",
    },
    title=f"{PRIMARY_SCENARIO}: solar vs wind availability per 1° cell",
    hover_data=["latitude", "longitude", "max_capacity_mw"],
    opacity=0.4,
)
fig_avail.add_shape(
    type="line",
    x0=0,
    x1=1,
    y0=0,
    y1=1,
    line=dict(dash="dash", color="grey"),
)
fig_avail.show()

In [ ]:
primary_plot_df = max_caps_df.loc[
    max_caps_df[MAP_METRIC].notna() & (max_caps_df[MAP_METRIC] > 0),
    [
        "latitude",
        "longitude",
        MAP_METRIC,
        "onshore_land_pct",
        "max_power_solar_mw",
        "max_power_wind_mw",
    ],
].copy()

fig = build_heatmap(
    primary_plot_df,
    MAP_METRIC,
    title=f"{PRIMARY_SCENARIO}: {MAP_METRIC}",
    color_scale="Viridis",
    cell_size_deg=CELL_SIZE_DEG,
    hover_cols=[
        "latitude",
        "longitude",
        "onshore_land_pct",
        "max_power_solar_mw",
        "max_power_wind_mw",
    ],
)
fig.show()

In [ ]:
if comparison_delta_df is None:
    print("No comparison scenario selected; load at least two scenario CSVs to draw a delta heatmap.")
else:
    delta_plot_df = comparison_delta_df.loc[
        comparison_delta_df["metric_delta"].abs() > 1e-9,
        [
            "latitude",
            "longitude",
            "metric_delta",
            "onshore_land_pct_target",
            f"{DIFFERENCE_METRIC}_reference",
            f"{DIFFERENCE_METRIC}_target",
        ],
    ].rename(columns={"onshore_land_pct_target": "onshore_land_pct"})

    if delta_plot_df.empty:
        print("The selected scenarios are identical for the chosen metric, so there is no delta heatmap to draw.")
    else:
        fig = build_heatmap(
            delta_plot_df,
            "metric_delta",
            title=f"{COMPARISON_TARGET} - {PRIMARY_SCENARIO}: {DIFFERENCE_METRIC}",
            color_scale="RdBu",
            color_midpoint=0.0,
            cell_size_deg=CELL_SIZE_DEG,
            hover_cols=[
                "latitude",
                "longitude",
                "onshore_land_pct",
                f"{DIFFERENCE_METRIC}_reference",
                f"{DIFFERENCE_METRIC}_target",
            ],
        )
        fig.show()

## Optional export

This section writes the current comparison outputs to `results/max_capacities_analysis/`.

It saves:
- summary CSVs for the loaded scenarios
- the primary-scenario stats tables
- the comparison summary tables when a comparison is active
- HTML figure exports for the availability scatter, primary heatmap, and delta heatmap when available

The export tag is built from the current `PRIMARY_SCENARIO` and `COMPARISON_TARGET` settings, so you can rerun the notebook with a different scenario pair and keep the outputs separate.